# sensa-ai: Colab'da eğitim ve ölçüm (yalnız sayı)

**Güvenlik (D-001):** Bu defter hiçbir görüntü göstermez. Soldaki **Dosyalar** panelinde `data/pozitif` klasörünü açma, görüntülere tıklama (Colab önizleme açar). Drive'a veri yalnız tek bir `.tar` dosyası olarak gider (Drive tar içini önizlemez). Çıktılar yalnız tablo ve sayı.

Başlamadan: Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**. Sol menüde 🔑 **Gizli anahtarlar** → `ROBOFLOW_API_KEY` ekle ve bu deftere erişim ver.

**Her yeni oturumda yukarıdan aşağı çalıştır.** Colab oturumu kapanınca her şey silinir; model, veri arşivi ve sonuçlar Drive'da (`MyDrive/sensa_ai/`) kalır.

In [ ]:
import os, subprocess
from pathlib import Path
def sh(cmd):
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)
os.chdir('/content')
if Path('sensa-ai').exists():
    sh('git -C sensa-ai pull -q')
else:
    sh('git clone -q -b claude/repo-yapisi-ve-plan https://github.com/sulku-AI/sensa-ai.git')
os.chdir('/content/sensa-ai')
sh('pip install -q -e ".[models,data,train]" sentencepiece protobuf')
sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/sensa_ai')
(DRIVE / 'sonuclar').mkdir(parents=True, exist_ok=True)

## 1. Veri

Drive'da arşiv varsa oradan açar (birkaç dakika). Yoksa Roboflow ve COCO'dan indirir ve arşivi Drive'a yazar (ilk sefer ~30 dk).

In [ ]:
ARSIV = DRIVE / 'veri.tar'
if ARSIV.exists():
    sh(f'tar -xf "{ARSIV}"')
else:
    from google.colab import userdata
    os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
    sh('python -m tools.fetch_datasets blood_seg_joost blood_pure_blur')
    sh('python -m tools.fetch_negatives --per-class 40')
    sh('python -m tools.fetch_negatives --set zor --per-class 30')
    sh(f'tar -cf "{ARSIV}" data/pozitif/blood_seg_joost data/pozitif/blood_pure_blur data/negatif')
if not Path('data/negatif/coco_zor').exists():
    sh('python -m tools.fetch_negatives --set zor --per-class 30')
sh('python -m tools.train_rfdetr prepare --src data/pozitif/blood_seg_joost data/pozitif/blood_pure_blur --neg data/negatif/coco --aug-per-image 2')

## 1b. Veri denetimi (görüntü göstermez)

Sayılar, kopyalar ve modelle en çok çelişen 20 görüntünün listesi. Sonuç Drive'a da yazılır.

In [ ]:
CK = DRIVE / 'rfdetr' / 'checkpoint_best_total.pth'
sh(f'python -m tools.audit_dataset data/pozitif/blood_pure_blur --detector rfdetr:{CK} --device cuda --worst 20 --list "{DRIVE}/denetim_pure_blur.json"')
sh(f'cp experiments/results/*_denetim.md "{DRIVE}/sonuclar/"')

## 2. Eğitim (RF-DETR-Seg Nano)

İlk model zaten eğitildi (`MyDrive/sensa_ai/rfdetr/checkpoint_best_total.pth`). Yeniden eğitmek için `EGIT = True` yap. T4'te 20 tur ~3 saat; sekme açık kalsın.

In [ ]:
EGIT = False
OUT = DRIVE / 'rfdetr'
if EGIT:
    sh(f'python -m tools.train_rfdetr train --epochs 20 --batch 8 --accum 2 --device cuda --out "{OUT}"')
CK = OUT / 'checkpoint_best_total.pth'
print('model:', CK, CK.exists())

## 3. Ölçüm (sonuçlar Drive'a da yazılır)

`test`: hiç görülmedi. `valid`: model seçiminde kullanıldı (hafif iyimser) ama 8 kat büyük. `coco_zor`: eğitimde olmayan insanlı ve kırmızı nesneli kansız sahneler (yanlış alarm sınaması).

In [ ]:
R = f'--detector rfdetr:{CK} --no-gate --device cuda --results "{DRIVE}/sonuclar"'
sh(f'python -m tools.evaluate --pos data/pozitif/egitim/test --empty-as-neg --box-threshold 0.3 {R}')
sh(f'python -m tools.evaluate --pos data/pozitif/egitim/valid --empty-as-neg --box-threshold 0.3 {R}')
sh(f'python -m tools.evaluate --neg data/negatif/coco_zor --box-threshold 0.3 {R}')
sh(f'python -m tools.evaluate --neg data/negatif/coco_zor --box-threshold 0.5 {R}')

## 4. Hız (T4)

In [ ]:
sh(f'python -m tools.bench --detector rfdetr:{CK} --res 720 --frames 20 --no-gate --device cuda')